<a href="https://colab.research.google.com/github/LuciAguiar/Tech_Chalenge_Obesity/blob/main/Obesity_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [4]:
# ==========================================
# 1. LEITURA DOS DADOS
# ==========================================
# Lembre-se de fazer o upload do arquivo 'Obesity.csv' no Colab antes de rodar!
df = pd.read_csv('Obesity.csv', dtype=str, sep=';', encoding='latin1')
df.columns = df.columns.str.strip()

# ==========================================
# 2. LIMPEZA INTELIGENTE (Winsorização / Clip)
# ==========================================
idade_num = pd.to_numeric(df['Age'].str.split('.').str[0], errors='coerce')
df['Age'] = idade_num.clip(lower=14, upper=61).fillna(idade_num.median())

def limpar_categorica_clip(coluna_nome, limite_inf, limite_sup):
    num = pd.to_numeric(df[coluna_nome], errors='coerce').round()
    num_clipado = num.clip(lower=limite_inf, upper=limite_sup)
    return num_clipado.fillna(num_clipado.mode()[0])

df['FCVC'] = limpar_categorica_clip('FCVC', 1, 3)
df['NCP']  = limpar_categorica_clip('NCP', 1, 4)
df['CH2O'] = limpar_categorica_clip('CH2O', 1, 3)
df['FAF']  = limpar_categorica_clip('FAF', 0, 3)
df['TUE']  = limpar_categorica_clip('TUE', 0, 2)

colunas_inteiras = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
for column in colunas_inteiras:
    df[column] = df[column].astype('Int64')

# Preenche valores em branco residuais com a resposta mais comum (moda)
df = df.fillna(df.mode().iloc[0])

# ==========================================
# 3. PREPARAÇÃO DO MODELO (Perfil Comportamental)
# ==========================================
# Removendo Height e Weight para evitar o Data Leakage da fórmula do IMC
X = df.drop(['Obesity', 'Height', 'Weight'], axis=1)
y = df['Obesity']

# One-Hot Encoding para as variáveis de texto (ex: transformar Gender em colunas numéricas)
X_num = pd.get_dummies(X, drop_first=True)

# Transformação das 7 categorias de obesidade em números (0 a 6)
le = LabelEncoder()
y_num = le.fit_transform(y)

# ==========================================
# 4. DIVISÃO E TREINAMENTO
# ==========================================
X_treino, X_teste, y_treino, y_teste = train_test_split(X_num, y_num, test_size=0.3, random_state=42)

modelo_rf = RandomForestClassifier(random_state=42)
modelo_rf.fit(X_treino, y_treino)

# ==========================================
# 5. AVALIAÇÃO DE PERFORMANCE
# ==========================================
previsoes = modelo_rf.predict(X_teste)
acuracia = accuracy_score(y_teste, previsoes)

print("="*60)
print(f"✅ Treinamento Concluído com Sucesso!")
print(f"📊 Base de dados mantida intacta com {len(df)} pacientes.")
print(f"🎯 Acurácia do Modelo Comportamental: {acuracia * 100:.2f}%\n")
print("📋 Relatório de Classificação Detalhado:")

# Recuperando os nomes originais para imprimir um relatório legível
nomes_classes = le.inverse_transform(range(len(le.classes_)))
print(classification_report(y_teste, previsoes, target_names=nomes_classes))
print("="*60)

# ==========================================
# 6. GERAÇÃO DOS BINÁRIOS (.pkl)
# ==========================================
joblib.dump(modelo_rf, 'modelo_obesidade.pkl')
joblib.dump(le, 'label_encoder.pkl')
joblib.dump(list(X_treino.columns), 'colunas_modelo.pkl')

print("💾 Arquivos .pkl gerados! Atualize a aba de arquivos à esquerda para fazer o download.")

✅ Treinamento Concluído com Sucesso!
📊 Base de dados mantida intacta com 2111 pacientes.
🎯 Acurácia do Modelo Comportamental: 73.50%

📋 Relatório de Classificação Detalhado:
                     precision    recall  f1-score   support

Insufficient_Weight       0.76      0.87      0.81        86
      Normal_Weight       0.64      0.61      0.63        93
     Obesity_Type_I       0.71      0.66      0.68       102
    Obesity_Type_II       0.71      0.82      0.76        88
   Obesity_Type_III       0.94      0.96      0.95        98
 Overweight_Level_I       0.65      0.62      0.64        88
Overweight_Level_II       0.70      0.58      0.63        79

           accuracy                           0.74       634
          macro avg       0.73      0.73      0.73       634
       weighted avg       0.73      0.74      0.73       634

💾 Arquivos .pkl gerados! Atualize a aba de arquivos à esquerda para fazer o download.
